# LangGraph Module · Day 09 — Sub-graphs & Parallel Nodes

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Build a **parent → child sub-graph** architecture
- [ ] Run two research nodes **in parallel**
- [ ] Use the **`Send()` API** for dynamic fan-out (map-reduce)
- [ ] **Aggregate** results in a merge node
- [ ] Document the pattern for reuse

**Python skill that feeds this (Day 09):** `asyncio.gather()` — parallel nodes are LangGraph's version of "run independent work concurrently, then merge," exactly what `gather` does.


## 0. Setup

All of today runs with **no API key** — sub-graphs, parallel fan-out, `Send()`, and merging are **graph-structure** features, not model features. Nodes are plain functions so you can watch the wiring behave. The optional last section shows where a real LLM would slot in.

In [9]:
from dotenv import load_dotenv
import os
load_dotenv()
HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
print("sub-graphs & parallel nodes notebook")
print("GOOGLE_API_KEY:", "loaded ✅" if HAS_KEY else "MISSING ❌ (not needed — everything here is keyless)")

sub-graphs & parallel nodes notebook
GOOGLE_API_KEY: loaded ✅


## 1. Two ways to grow a graph — *nest* it and *widen* it

So far your graphs were a single flat chain of nodes. Real systems grow in two directions:

- **Sub-graphs (nesting)** — package a whole graph and drop it in as **one node** of a bigger graph. Reuse and encapsulation, like calling a function.
- **Parallel nodes (widening)** — run several nodes **at the same time** on independent work, then **merge** their outputs. Speed, like Day 09's `asyncio.gather`.

We'll build both around a banking job: **research a company for a credit decision** — pull from several sources at once, each source's logic packaged as a reusable sub-graph.

## 2. Sub-graph — a compiled graph used as a node

A compiled graph is just a callable with the same interface as a node (state in → state update out). So you can `add_node("research", child_graph)` and the parent treats the entire child as one step. Here's a small **research sub-graph**: fetch → summarise.

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict):
    source: str
    query: str
    summary: str

def fetch(s: ResearchState) -> dict:
    # pretend we hit an API for this source
    return {"query": f"{s['query']} [fetched from {s['source']}]"}

def summarise(s: ResearchState) -> dict:
    return {"summary": f"{s['source']}: 3 key findings on {s['query'].split(' [')[0]}"}

rb = StateGraph(ResearchState)
rb.add_node("fetch", fetch)
rb.add_node("summarise", summarise)
rb.add_edge(START, "fetch")
rb.add_edge("fetch", "summarise")
rb.add_edge("summarise", END)
research_graph = rb.compile()           # <- a reusable, self-contained unit

# it runs on its own, like any graph:
print(research_graph.invoke({"source": "news", "query": "HSBC", "summary": ""}))

{'source': 'news', 'query': 'HSBC [fetched from news]', 'summary': 'news: 3 key findings on HSBC'}


☝ `research_graph` is a **black box** with a clean contract: give it a `source` + `query`, get back a `summary`. It has its own internal nodes (`fetch` → `summarise`) that the outside world never sees. That encapsulation is the whole point of a sub-graph — build once, reuse anywhere, test in isolation. Next we run **several** of these at once.

## 3. Parallel nodes — fan out from `START`, collect with a reducer

To run nodes **in parallel**, give them a **common entry** (edges from `START` to each). LangGraph runs all nodes that are ready in the same **superstep** concurrently. They each write to the **same** state field, so that field needs a **reducer** (`operator.add`) to *append* instead of overwrite — otherwise the last writer wins and you lose the others.

In [11]:
import operator
from typing import Annotated

class CreditState(TypedDict):
    company: str
    findings: Annotated[list, operator.add]   # reducer: parallel writes APPEND
    verdict: str

def research_news(s: CreditState) -> dict:
    return {"findings": [f"news: {s['company']} sentiment stable"]}

def research_filings(s: CreditState) -> dict:
    return {"findings": [f"filings: {s['company']} debt ratio 0.4"]}

def research_market(s: CreditState) -> dict:
    return {"findings": [f"market: {s['company']} CDS spreads normal"]}

Now wire three research nodes to fan out from `START`, and a **merge** node they all flow into. Because the merge node has an edge from *each* branch, LangGraph waits for **all three** to finish before running it — an automatic barrier.

In [12]:
def merge(s: CreditState) -> dict:
    # runs only after ALL parallel branches complete; reads the collected list
    n = len(s["findings"])
    return {"verdict": f"APPROVE — reviewed {n} sources: " + " | ".join(s["findings"])}

cb = StateGraph(CreditState)
cb.add_node("research_news", research_news)
cb.add_node("research_filings", research_filings)
cb.add_node("research_market", research_market)
cb.add_node("merge", merge)

for n in ["research_news", "research_filings", "research_market"]:
    cb.add_edge(START, n)      # fan OUT: all three start together
    cb.add_edge(n, "merge")    # fan IN: merge waits for all three

cb.add_edge("merge", END)
credit_graph = cb.compile()

out = credit_graph.invoke({"company": "HSBC", "findings": [], "verdict": ""})
print("findings collected:", len(out["findings"]))
for f in out["findings"]:
    print("  -", f)
print("\nverdict:", out["verdict"])

findings collected: 3
  - filings: HSBC debt ratio 0.4
  - market: HSBC CDS spreads normal
  - news: HSBC sentiment stable

verdict: APPROVE — reviewed 3 sources: filings: HSBC debt ratio 0.4 | market: HSBC CDS spreads normal | news: HSBC sentiment stable


☝ Three branches ran in parallel and the reducer **appended** each one's finding into `findings`; `merge` fired once, after all three, and read the full list. This is the graph-level twin of `asyncio.gather` from today's Python skill: **independent work overlaps, then one node aggregates.** The reducer is the key detail — without `Annotated[list, operator.add]`, the three parallel writes would clobber each other down to one.

```mermaid
flowchart TD
    START([__start__]) --> research_news
    START --> research_filings
    START --> research_market
    research_news --> merge
    research_filings --> merge
    research_market --> merge
    merge --> END([__end__])
```

## 4. `Send()` — dynamic fan-out (map-reduce) over a list you don't know ahead of time

§3 fanned out to a **fixed** three nodes. But often the number of branches is **decided at runtime** — audit *N* accounts, research *N* competitors — where *N* comes from the state. The **`Send()`** API is the answer: a routing function returns a **list of `Send(node, state)`**, and LangGraph spins up **one parallel instance of that node per item**. That's the classic **map** (fan out over items) → **reduce** (merge with a reducer) pattern.

In [13]:
from langgraph.types import Send

class AuditState(TypedDict):
    accounts: list                              # the "map" input — size unknown until runtime
    reports: Annotated[list, operator.add]      # reducer collects each mapped result

def audit_one(s: dict) -> dict:
    # ONE instance runs per account, in parallel
    acc = s["account"]
    flag = "⚠️ REVIEW" if acc.endswith("9") else "ok"
    return {"reports": [f"account {acc}: {flag}"]}

def fan_out(s: AuditState):
    # the map step: emit one Send per account -> N parallel audit_one runs
    return [Send("audit_one", {"account": a}) for a in s["accounts"]]

ab = StateGraph(AuditState)
ab.add_node("audit_one", audit_one)
ab.add_conditional_edges(START, fan_out, ["audit_one"])   # START routes via fan_out
ab.add_edge("audit_one", END)
audit_graph = ab.compile()

result = audit_graph.invoke({"accounts": ["acc-101", "acc-209", "acc-333", "acc-449"], "reports": []})
for r in result["reports"]:
    print(r)

account acc-101: ok
account acc-209: ⚠️ REVIEW
account acc-333: ok
account acc-449: ⚠️ REVIEW


☝ Four accounts in → four **parallel** `audit_one` runs → results **reduced** into `reports`. Pass a list of 40 and you get 40 parallel runs with **no code change** — that's the power of `Send` over fixed edges: the fan-out width is **data-driven**. `Send(node, payload)` also lets you hand each branch its *own* mini-state (`{"account": a}`), so branches don't fight over shared fields.

## 5. Put it together — a sub-graph *inside* a parallel fan-out

The real reuse win: make each parallel branch a **sub-graph** instead of a bare function. Here the parent fans out to three instances of the §2 `research_graph`, each on a different source, then merges. Nesting + widening in one graph.

In [14]:
class PipelineState(TypedDict):
    company: str
    summaries: Annotated[list, operator.add]
    report: str

def run_source(source):
    # wrap the reusable research sub-graph as a branch node bound to one source
    def _node(s: PipelineState) -> dict:
        r = research_graph.invoke({"source": source, "query": s["company"], "summary": ""})
        return {"summaries": [r["summary"]]}
    return _node

def compile_report(s: PipelineState) -> dict:
    return {"report": f"CREDIT DOSSIER · {s['company']}\n  " + "\n  ".join(s["summaries"])}

pb = StateGraph(PipelineState)
for src in ["news", "filings", "market"]:
    pb.add_node(f"src_{src}", run_source(src))
    pb.add_edge(START, f"src_{src}")
    pb.add_edge(f"src_{src}", "compile_report")
pb.add_node("compile_report", compile_report)
pb.add_edge("compile_report", END)
pipeline = pb.compile()

out = pipeline.invoke({"company": "Barclays", "summaries": [], "report": ""})
print(out["report"])

CREDIT DOSSIER · Barclays
  filings: 3 key findings on Barclays
  market: 3 key findings on Barclays
  news: 3 key findings on Barclays


☝ Each branch is the **whole `research_graph` sub-graph** (fetch → summarise) running in parallel, and `compile_report` merges their summaries. This is the reusable architecture the plan asks for: **sub-graphs as building blocks, fanned out in parallel, aggregated by a merge node.** Swap the fake `fetch`/`summarise` for real API + LLM calls and the parent graph is unchanged.

## 6. Swap in a real LLM — the structure never changes (needs `GOOGLE_API_KEY`)

Only the *inside of one node* changes: `summarise` now calls Gemini instead of returning a canned string. We rebuild the research sub-graph with that real node, then run the **same** parallel fan-out pipeline from §5 — proving the wiring is model-agnostic. This runs live if a key is present.

In [15]:
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import HumanMessage
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

    # real-LLM version of the §2 summarise node — same in/out contract
    def summarise_real(s: ResearchState) -> dict:
        prompt = f"In one sentence, give a credit-risk read on {s['query']} from a {s['source']} angle."
        text = llm.invoke([HumanMessage(content=prompt)]).content.strip()
        return {"summary": f"{s['source']}: {text}"}

    # rebuild the sub-graph with the real node (fetch unchanged)
    rb2 = StateGraph(ResearchState)
    rb2.add_node("fetch", fetch)
    rb2.add_node("summarise", summarise_real)
    rb2.add_edge(START, "fetch")
    rb2.add_edge("fetch", "summarise")
    rb2.add_edge("summarise", END)
    research_graph_real = rb2.compile()

    # reuse the §5 parallel pipeline, but point branches at the real sub-graph
    def run_source_real(source):
        def _node(s: PipelineState) -> dict:
            r = research_graph_real.invoke({"source": source, "query": s["company"], "summary": ""})
            return {"summaries": [r["summary"]]}
        return _node

    pb2 = StateGraph(PipelineState)
    for src in ["news", "filings", "market"]:
        pb2.add_node(f"src_{src}", run_source_real(src))
        pb2.add_edge(START, f"src_{src}")
        pb2.add_edge(f"src_{src}", "compile_report")
    pb2.add_node("compile_report", compile_report)
    pb2.add_edge("compile_report", END)
    pipeline_real = pb2.compile()

    out = pipeline_real.invoke({"company": "Barclays", "summaries": [], "report": ""})
    print(out["report"])
else:
    print("No key -> skipped. The keyless nodes above already taught the full pattern.")

CREDIT DOSSIER · Barclays
  filings: Barclays' filings consistently highlight robust capital and liquidity, ongoing asset quality management, and strategic efforts to enhance profitability within a dynamic regulatory environment, all detailed through comprehensive risk disclosures.
  market: From a market angle, Barclays' credit risk is generally perceived as manageable, reflected in relatively stable CDS spreads and investment-grade ratings, supported by its diversified business and capital, though subject to broader economic and regulatory pressures.
  news: Barclays' credit risk profile appears stable, bolstered by strong retail banking performance from higher interest rates, but tempered by ongoing investment banking volatility and potential increases in loan loss provisions amid a challenging economic outlook.


☝ Real Gemini text now fills each parallel branch — yet **`run_source_real`, the fan-out edges, and `compile_report` are structurally identical to §5.** Only `summarise` changed. That's the Day 07/08 lesson again: **write the graph once; the model is a node's implementation detail.** Parallelism, sub-graphs, and merging are properties of the *graph*, unaffected by what a node does inside.

## 7. Recap — the reusable pattern

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **sub-graph** | a compiled graph used as one node of a parent | encapsulate & reuse a multi-step unit; test in isolation |
| **parallel nodes** | multiple edges from `START` (or one node) → same superstep | run independent work concurrently (graph-level `gather`) |
| **reducer** `Annotated[list, operator.add]` | merges concurrent writes to one field | parallel branches **append** instead of overwriting |
| **merge node** | a node all branches flow into | barrier: waits for all, then aggregates the collected list |
| **`Send(node, state)`** | dynamic fan-out from a routing fn | one parallel branch **per item** — width decided at runtime (map-reduce) |

**Document-for-reuse rule of thumb:**
1. Package each repeatable multi-step job as a **sub-graph** with a clean state contract.
2. **Fan out** independent work from `START` (fixed count) or via **`Send`** (runtime count).
3. Give every field written in parallel a **reducer**.
4. **Merge** in one node that reads the collected list and produces the final output.

**Takeaway:** nest to reuse, widen to go fast. Parallel nodes + merge is `asyncio.gather` at the graph level — today's Python skill, wearing a LangGraph hat.

